<a href="https://colab.research.google.com/github/ksusmitha879-cyber/FlyrankStarter/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ksusmitha879-cyber/FlyrankStarter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**My answer: Ranking / scoring.**

My lane's question is "which visible pages should a reviewer look at *first*?" — that is
literally the "Which ones first?" pattern from the framing guide, which maps to
ranking/scoring, not classification. I am not trying to predict a fixed yes/no label for every
page; I want a continuous priority score I can sort by, so a reviewer with limited time works
down the list from the top. Section 4 (unit of analysis) and the code below both operate at
the level of "one score per page," not "one class per page."
*Classification, clustering, ranking, or scoring — which one, and why?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv('/content/content_refresh_anonymized.csv')
valid = df[df['avg_position'] > 0].copy()  # avg_position == 0 means "no position data"

# A ranking/scoring task needs a reasonably large pool to rank WITHIN each comparison group
# (here: position tier). Quick check that every tier has enough pages to make "rank within
# tier" meaningful, not just a handful of pages per group.
print("Rows with real position data:", len(valid), "of", len(df))
print()
print(valid['position_tier'].value_counts())

Rows with real position data: 28795 of 30000

position_tier
page_1      11814
striking     7304
page_3_5     7242
deep         1319
top_3        1116
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
**My answer: proxy, not a fully observed target — and I want to be explicit about that.**

What I would score is **estimated lost clicks**: for each visible page, how many extra clicks
it would likely have earned over the last 90 days if its CTR simply matched the *median* CTR
of other pages in its own position tier. In code:

`estimated_lost_clicks = max(0, tier_median_ctr - page_ctr) / 100 * impressions_90d`

This is a **proxy**, not an observed outcome, per the framing skill's own warning: it comes
from a rule I defined (compare to the tier median), not from watching what actually happened
after someone fixed a page's title or meta description. There is no ground-truth "this page
would have earned exactly N more clicks" label anywhere in the data — nobody has run that
experiment yet. I am using this proxy because it is the best available stand-in for "how much
is at stake here," and Section 5 below (Why ML beats a fixed rule) also uses an *independent*
observed signal (`trend_direction`) as a partial sanity check on this proxy, precisely because
I don't fully trust a rule-defined target on its own.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tier_median_ctr = valid.groupby('position_tier')['ctr'].median()
valid['tier_median_ctr'] = valid['position_tier'].map(tier_median_ctr)
valid['opportunity_gap'] = (valid['tier_median_ctr'] - valid['ctr']).clip(lower=0)

# Only score pages with enough traffic to be worth a reviewer's time
visible = valid[valid['impressions_90d'] >= 500].copy()
visible['est_lost_clicks'] = visible['opportunity_gap'] / 100 * visible['impressions_90d']

print(f"Visible pages (impressions_90d >= 500): {len(visible):,}")
visible[['content_id', 'position_tier', 'ctr', 'tier_median_ctr',
         'impressions_90d', 'est_lost_clicks']].sort_values(
    'est_lost_clicks', ascending=False).head(10)

Visible pages (impressions_90d >= 500): 16,726


,content_id,position_tier,ctr,tier_median_ctr,impressions_90d,est_lost_clicks
7445,content_c8e9d6ab9013,page_1,0.00,0.16,208678,333.8848
3394,content_36ff89c8214e,page_1,0.05,0.16,295097,324.6067
6903,content_c84a0ab98e90,page_1,0.03,0.16,223271,290.2523
27178,content_453722754fea,page_1,0.01,0.16,140079,210.1185
9193,content_c1fe78bc4e37,page_1,0.03,0.16,134055,174.2715
482,content_39881853ef0c,page_1,0.01,0.16,112434,168.6510
15914,content_0919dd345d80,page_1,0.02,0.16,119217,166.9038
4708,content_b115f7c74779,page_1,0.03,0.16,123469,160.5097
3070,content_91652435f57a,page_1,0.06,0.16,159590,159.5900
5621,content_97a86caf3a3d,page_1,0.07,0.16,147670,132.9030


## 3. Success metric

*One metric you can defend. What number means 'good'?*
**My answer: precision@20.**

Per the framing guide's own table, ranking/scoring tasks are measured with precision@K, and
K=20 matches the real constraint from Section 2 of last week's notebook: a reviewer only has
time to look at a handful of pages a week, so what matters is whether the pages at the *top*
of my list are genuinely worth their time — not how the whole 16,000-page list is ordered.

I can't fully compute true precision@20 yet, because that requires an editor to actually look
at the top 20 and judge whether each one is a real, fixable opportunity — a label I don't have.
What I *can* compute today, as a first honest check, is whether my ranked top-20 skews toward
pages that are *independently* observed to be losing traffic (`trend_direction == 'down'`,
which comes from real period-over-period session/impression counts, not from my scoring rule
at all). That's not the real metric, but it's a today-computable sanity check in the same
spirit.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = visible.sort_values('est_lost_clicks', ascending=False).head(20)

base_rate_down = (visible['trend_direction'] == 'down').mean()
top20_rate_down = (top20['trend_direction'] == 'down').mean()

print(f"Base rate of trend_direction == 'down' among all visible pages: {base_rate_down:.3f}")
print(f"Rate of trend_direction == 'down' among my top-20 ranked pages:  {top20_rate_down:.3f}")
print()
print("This is an honest, slightly humbling result: my top-20 list does NOT clearly skew")
print("toward pages that are independently observed to be declining. A single-formula proxy")
print("(gap x impressions) isn't obviously capturing the same thing as recent trend -- which")
print("is itself evidence for Section 5 below.")

Base rate of trend_direction == 'down' among all visible pages: 0.596
Rate of trend_direction == 'down' among my top-20 ranked pages:  0.550

This is an honest, slightly humbling result: my top-20 list does NOT clearly skew
toward pages that are independently observed to be declining. A single-formula proxy
(gap x impressions) isn't obviously capturing the same thing as recent trend -- which
is itself evidence for Section 5 below.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
**My answer: one row = one content page, scored over its most recent 90-day window.**

Not a client, not a day — a single piece of published content (`content_id`), because the
decision I'm supporting ("which page should a reviewer open first") is made at the page level.
The dataframe below shows exactly that: one row per page, with the columns a reviewer would
actually need to act — its position tier (who it's being compared against), its real CTR, its
tier peers' median CTR, how much traffic it gets, and my proxy score.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_of_analysis = top20[['content_id', 'position_tier', 'main_intent',
                          'ctr', 'tier_median_ctr', 'impressions_90d',
                          'est_lost_clicks', 'trend_direction']].reset_index(drop=True)
unit_of_analysis

,content_id,position_tier,main_intent,ctr,tier_median_ctr,impressions_90d,est_lost_clicks,trend_direction
0,content_c8e9d6ab9013,page_1,informational,0.00,0.16,208678,333.8848,down
1,content_36ff89c8214e,page_1,informational,0.05,0.16,295097,324.6067,stable
2,content_c84a0ab98e90,page_1,informational,0.03,0.16,223271,290.2523,stable
3,content_453722754fea,page_1,informational,0.01,0.16,140079,210.1185,down
4,content_c1fe78bc4e37,page_1,commercial,0.03,0.16,134055,174.2715,down
5,content_39881853ef0c,page_1,informational,0.01,0.16,112434,168.6510,down
6,content_0919dd345d80,page_1,informational,0.02,0.16,119217,166.9038,down
7,content_b115f7c74779,page_1,transactional,0.03,0.16,123469,160.5097,up
8,content_91652435f57a,page_1,commercial,0.06,0.16,159590,159.5900,stable
9,content_97a86caf3a3d,page_1,transactional,0.07,0.16,147670,132.9030,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
**My answer: because "what counts as low CTR" moves under at least three signals at once,
and a single hand-picked threshold can't track all of them together.**

Evidence 1 — even within the *same* position tier, expected CTR shifts with intent. Within
`page_1` alone, median CTR is 0.14% for informational pages, 0.16% for commercial, and 0.21%
for transactional (code below). A single per-tier cutoff already misses this.

Evidence 2 — my own proxy score (gap x impressions) does not obviously line up with the
independently observed decline signal (Section 3's check above: 55% of my top-20 are trending
down, versus a 60% base rate — no clear lift). That means "big static gap" and "currently
declining" are not the same thing, and a fixed rule built around just one of them would miss
real opportunities the other one catches. A model that learns how tier, intent, content type,
freshness, and trend combine — rather than a rule I hand-pick along one axis — has a real
chance of capturing that interaction. A plain if-statement does not.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
page1 = valid[valid['position_tier'] == 'page_1']
print("Median CTR by intent, WITHIN the page_1 tier only:")
print(page1.groupby('main_intent')['ctr'].agg(['count', 'median', 'mean']).round(3))

Median CTR by intent, WITHIN the page_1 tier only:
               count  median   mean
main_intent                        
commercial      1757    0.16  0.352
informational   6641    0.14  0.429
navigational      13    0.32  3.988
transactional   2422    0.21  0.354
